# Amazon ML Challenge — Matcher V1

## Objective

Build the first baseline entity-matching model using labeled S1–S2/S3
candidate pairs.

### Input

A candidate-pair table containing:

- `source1_entity_id`
- `candidate_entity_id`

with the candidate entity's source record available for feature generation.

### Output

A match probability for each candidate pair:

- `1` → likely same business
- `0` → likely different businesses

The matcher will not enforce one-to-one matching. Multiple candidates may
be predicted as matches for the same S1 entity, and an S1 entity may have
zero predicted matches.

### Day 1 Goal

Establish a reproducible baseline matcher and evaluate it using
entity-level macro F0.5 on the held-out validation S1 entities.

## 1. Matcher Interface

The matcher consumes the final candidate set produced by the candidate
generation stage.

Required candidate-pair columns:

- `source1_entity_id`
- `candidate_entity_id`

The candidate entity ID must refer to an existing S2 or S3 record.

The matcher is responsible for:

1. Constructing pair-level features
2. Assigning training labels from ground truth
3. Training a baseline binary classifier
4. Producing match probabilities
5. Selecting a decision threshold
6. Evaluating predictions using entity-level macro F0.5

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score,
    classification_report
)

RANDOM_STATE = 42
CHUNK_SIZE = 100_000

In [2]:
PROJECT_ROOT = Path("..")

DATASET_DIR = PROJECT_ROOT / "dataset"
TRAIN_DIR = DATASET_DIR / "train"

TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
TRAIN_GT = TRAIN_DIR / "train_ground_truth.tsv"

print("S1:", TRAIN_S1.exists())
print("S2:", TRAIN_S2.exists())
print("S3:", TRAIN_S3.exists())
print("GT:", TRAIN_GT.exists())

S1: True
S2: True
S3: True
GT: True


In [3]:
from sklearn.model_selection import train_test_split

gt = pd.read_csv(
    TRAIN_GT,
    sep="\t",
    keep_default_na=False
)

all_s1_ids = gt["source1_entity_id"].values

train_s1_ids, valid_s1_ids = train_test_split(
    all_s1_ids,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_s1_set = set(train_s1_ids)
valid_s1_set = set(valid_s1_ids)

print("Training S1:", len(train_s1_set))
print("Validation S1:", len(valid_s1_set))

assert not (train_s1_set & valid_s1_set)

print("✓ Entity-level split reproduced successfully.")

Training S1: 1765456
Validation S1: 441365
✓ Entity-level split reproduced successfully.


In [4]:
DEV_S1_SIZE = 5_000

rng = np.random.default_rng(RANDOM_STATE)

dev_s1_ids = rng.choice(
    np.array(list(train_s1_set)),
    size=DEV_S1_SIZE,
    replace=False
)

dev_s1_ids = set(dev_s1_ids)

print("Development S1 entities:", len(dev_s1_ids))

Development S1 entities: 5000


In [5]:
dev_gt = gt[
    gt["source1_entity_id"].isin(dev_s1_ids)
].copy()

dev_gt["matched_ids"] = dev_gt["matched_entity_ids"].apply(
    lambda x: [] if x.strip() == "" else x.split(",")
)

print("Development GT rows:", len(dev_gt))

print(
    "S1 entities with at least one true match:",
    (dev_gt["matched_entity_ids"].str.strip() != "").sum()
)

print(
    "S1 entities with no true match:",
    (dev_gt["matched_entity_ids"].str.strip() == "").sum()
)

Development GT rows: 5000
S1 entities with at least one true match: 4714
S1 entities with no true match: 286


In [6]:
positive_pairs = []

for _, row in dev_gt.iterrows():
    s1_id = row["source1_entity_id"]

    for candidate_id in row["matched_ids"]:
        positive_pairs.append({
            "source1_entity_id": s1_id,
            "candidate_entity_id": candidate_id,
            "label": 1
        })

positive_pairs = pd.DataFrame(positive_pairs)

print("Positive pairs:", len(positive_pairs))
display(positive_pairs.head(10))

Positive pairs: 17603


,source1_entity_id,candidate_entity_id,label
0,S1-731542669,S2-939487122,1
1,S1-731542669,S2-403134793,1
2,S1-731542669,S3-576199259,1
3,S1-731542669,S3-535243754,1
4,S1-731542669,S3-988120505,1
5,S1-153559765,S3-927742738,1
6,S1-153559765,S3-25113409,1
7,S1-153559765,S3-760336227,1
8,S1-713405718,S2-979692389,1
9,S1-713405718,S2-496110983,1


In [7]:
positive_count = len(positive_pairs)

print(f"Positive pairs: {positive_count:,}")

Positive pairs: 17,603


## 2. Temporary Development Negatives

The final candidate set will eventually come from the blocking pipeline.

Because the blocking pipeline is not available yet, we create a temporary
development-only negative set for Matcher V1 development.

Negative candidates are sampled from the same country as the S1 entity.
This avoids making the baseline task artificially easy through country
mismatch alone.

These temporary candidates will NOT be used as the final
`candidate_pairs.tsv`.

In [8]:
def retrieve_records_by_ids(
    path,
    entity_ids,
    chunksize=CHUNK_SIZE
):
    entity_ids = set(entity_ids)
    results = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize
    ):
        matched = chunk[
            chunk["entity_id"].isin(entity_ids)
        ]

        if not matched.empty:
            results.append(matched)

        if sum(len(x) for x in results) >= len(entity_ids):
            break

    if not results:
        return pd.DataFrame()

    return pd.concat(
        results,
        ignore_index=True
    )

In [9]:
dev_s1 = retrieve_records_by_ids(
    TRAIN_S1,
    dev_s1_ids
)

print("Retrieved S1 records:", len(dev_s1))
display(dev_s1.head())

Retrieved S1 records: 5000


,entity_id,business_name,business_address,country
0,S1-563661207,Rockstar & Sons of Pune City,"A 802 Wisteria, Keshav Nagar Manjri Rd., Pune ...",India
1,S1-610162639,Apex Textile Associates Inc,"7 Tanglewood Lane, New Milford, CT",US
2,S1-47756016,Best Plantations Private Limited,"Plot No 146, Pi Estate, Ghaziabad, Uttar Pradesh",India
3,S1-643318152,Dynamic Global Private Limited,"308, Apra North Ex Plaza, Plot No.H-2, Netaji ...",India
4,S1-477299338,Pioneer Healthcare Private Limited,"Maharashtra, Mumbai, Sahakar Bhavan Kathabazar...",India


## 3. Temporary Candidate Pool

The real candidate pool will eventually be produced by the teammate's
blocking pipeline.

For Matcher V1 development, we construct a temporary pool of S2/S3 records
from the same countries as the development S1 entities.

This pool is used only to generate development negatives and will not become
the final `candidate_pairs.tsv`.

In [10]:
DEV_POOL_SIZE = 100_000

dev_countries = set(dev_s1["country"].dropna().unique())

print("Development countries:", dev_countries)

Development countries: {'US', 'India'}


In [11]:
def sample_records_by_country(
    path,
    countries,
    target_size=100_000,
    chunksize=CHUNK_SIZE,
    random_state=RANDOM_STATE
):
    rng = np.random.default_rng(random_state)
    samples = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize
    ):
        chunk = chunk[
            chunk["country"].isin(countries)
        ]

        if not chunk.empty:
            samples.append(chunk)

    pool = pd.concat(
        samples,
        ignore_index=True
    )

    if len(pool) > target_size:
        indices = rng.choice(
            len(pool),
            size=target_size,
            replace=False
        )
        pool = pool.iloc[indices].reset_index(drop=True)

    return pool

In [12]:
s2_pool = sample_records_by_country(
    TRAIN_S2,
    dev_countries,
    target_size=DEV_POOL_SIZE,
    random_state=RANDOM_STATE
)

s3_pool = sample_records_by_country(
    TRAIN_S3,
    dev_countries,
    target_size=DEV_POOL_SIZE,
    random_state=RANDOM_STATE + 1
)

candidate_pool = pd.concat(
    [s2_pool, s3_pool],
    ignore_index=True
)

print("S2 pool:", len(s2_pool))
print("S3 pool:", len(s3_pool))
print("Combined candidate pool:", len(candidate_pool))

display(candidate_pool.head())

S2 pool: 100000
S3 pool: 100000
Combined candidate pool: 200000


,entity_id,business_name,business_address,country
0,S2-796341195,"GONZALEZ, BONITA DISCIPLINED PARTNERS","71-D FLETCHER STREET, GOSHEN, NY",US
1,S2-482237115,Blanchard Higth Center,"2520 PLAZA DR, TEMPE, AZ",US
2,S2-508345922,(india) Tfs Másale,"1 , SHOP NO. 3, VASUDEV INDUSTRIES ESTATE, KAL...",India
3,S2-316521483,"Empire Dymic Sachem, Holdings","11103 CARTER CANYON ROAD, MOUNT LEMMON, AZ",US
4,S2-234103134,குஜராத் யுனைடெட் பிசினஸ் பிரைவேட் லிமிடெட்,"D NO: 60/1/2 VELLIKALMEDU, ILAIYANELLORE POST,...",India


## 4. Temporary Negative Pair Generation

For Matcher V1 development, create non-matching S1–S2/S3 pairs from the
same country.

True ground-truth matches are explicitly excluded.

This is a temporary development dataset only. The final candidate set will
later be supplied by the blocking pipeline.

In [13]:
NEGATIVE_RATIO = 1.0

# Map each S1 entity to its true matched IDs
true_matches = {
    row["source1_entity_id"]: set(row["matched_ids"])
    for _, row in dev_gt.iterrows()
}

# Candidate pool grouped by country
candidate_by_country = {
    country: group.reset_index(drop=True)
    for country, group in candidate_pool.groupby("country")
}

rng = np.random.default_rng(RANDOM_STATE)

negative_pairs = []

for _, s1_row in dev_s1.iterrows():
    s1_id = s1_row["entity_id"]
    country = s1_row["country"]

    pool = candidate_by_country.get(country)

    if pool is None or len(pool) == 0:
        continue

    forbidden = true_matches.get(s1_id, set())

    # Sample a few candidates and keep only non-matches
    sample_size = 10

    indices = rng.integers(
        0,
        len(pool),
        size=sample_size
    )

    selected = pool.iloc[indices]

    for _, candidate in selected.iterrows():

        candidate_id = candidate["entity_id"]

        if candidate_id in forbidden:
            continue

        negative_pairs.append({
            "source1_entity_id": s1_id,
            "candidate_entity_id": candidate_id,
            "label": 0
        })

        if len(negative_pairs) >= int(
            len(positive_pairs) * NEGATIVE_RATIO
        ):
            break

    if len(negative_pairs) >= int(
        len(positive_pairs) * NEGATIVE_RATIO
    ):
        break

negative_pairs = pd.DataFrame(negative_pairs)

print(f"Positive pairs: {len(positive_pairs):,}")
print(f"Negative pairs: {len(negative_pairs):,}")

Positive pairs: 17,603
Negative pairs: 17,603


In [14]:
negative_check = []

for _, row in negative_pairs.iterrows():
    true_ids = true_matches.get(
        row["source1_entity_id"],
        set()
    )

    negative_check.append(
        row["candidate_entity_id"] in true_ids
    )

print(
    "Incorrect negative pairs:",
    sum(negative_check)
)

assert sum(negative_check) == 0

print("✓ No ground-truth positive pair was included as a negative.")

Incorrect negative pairs: 0
✓ No ground-truth positive pair was included as a negative.


In [15]:
dev_pairs = pd.concat(
    [
        positive_pairs,
        negative_pairs
    ],
    ignore_index=True
)

# Shuffle
dev_pairs = dev_pairs.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("Development pair dataset:", len(dev_pairs))

display(dev_pairs.head(10))

print("\nLabel distribution:")
display(
    dev_pairs["label"]
    .value_counts()
    .sort_index()
)

Development pair dataset: 35206


,source1_entity_id,candidate_entity_id,label
0,S1-589179138,S3-570768070,1
1,S1-721348248,S2-778835740,1
2,S1-841452359,S3-492698045,1
3,S1-371936370,S3-638288652,0
4,S1-466333379,S3-506627624,1
5,S1-74837897,S3-959247066,1
6,S1-25983407,S3-227654588,0
7,S1-447471479,S2-211749645,1
8,S1-758288766,S3-429961399,0
9,S1-712770952,S3-291728652,0



Label distribution:


label
0    17603
1    17603
Name: count, dtype: int64

## 5. Retrieve Pair Records

Retrieve the S1, S2, and S3 records required by the development pair
dataset.

Only records appearing in the development pairs are retrieved; the complete
source datasets are not loaded into memory.

In [16]:
# Collect all S1 IDs used in the development pairs
pair_s1_ids = set(dev_pairs["source1_entity_id"])

# Split candidate IDs by source
pair_s2_ids = set(
    dev_pairs.loc[
        dev_pairs["candidate_entity_id"].str.startswith("S2-"),
        "candidate_entity_id"
    ]
)

pair_s3_ids = set(
    dev_pairs.loc[
        dev_pairs["candidate_entity_id"].str.startswith("S3-"),
        "candidate_entity_id"
    ]
)

print("S1 IDs needed:", len(pair_s1_ids))
print("S2 IDs needed:", len(pair_s2_ids))
print("S3 IDs needed:", len(pair_s3_ids))

S1 IDs needed: 4811
S2 IDs needed: 16923
S3 IDs needed: 17551


In [17]:
s1_pair_records = retrieve_records_by_ids(
    TRAIN_S1,
    pair_s1_ids
)

s2_pair_records = retrieve_records_by_ids(
    TRAIN_S2,
    pair_s2_ids
)

s3_pair_records = retrieve_records_by_ids(
    TRAIN_S3,
    pair_s3_ids
)

candidate_pair_records = pd.concat(
    [
        s2_pair_records,
        s3_pair_records
    ],
    ignore_index=True
)

print("S1 records retrieved:", len(s1_pair_records))
print("S2 records retrieved:", len(s2_pair_records))
print("S3 records retrieved:", len(s3_pair_records))
print("Candidate records retrieved:", len(candidate_pair_records))

S1 records retrieved: 4811
S2 records retrieved: 16923
S3 records retrieved: 17551
Candidate records retrieved: 34474


In [18]:
assert len(s1_pair_records) == len(pair_s1_ids), (
    "Some required S1 records were not retrieved."
)

assert len(s2_pair_records) == len(pair_s2_ids), (
    "Some required S2 records were not retrieved."
)

assert len(s3_pair_records) == len(pair_s3_ids), (
    "Some required S3 records were not retrieved."
)

print("✓ All records required for Matcher V1 were retrieved.")

✓ All records required for Matcher V1 were retrieved.


In [19]:
display(s1_pair_records.head())
display(candidate_pair_records.head())

,entity_id,business_name,business_address,country
0,S1-563661207,Rockstar & Sons of Pune City,"A 802 Wisteria, Keshav Nagar Manjri Rd., Pune ...",India
1,S1-610162639,Apex Textile Associates Inc,"7 Tanglewood Lane, New Milford, CT",US
2,S1-47756016,Best Plantations Private Limited,"Plot No 146, Pi Estate, Ghaziabad, Uttar Pradesh",India
3,S1-643318152,Dynamic Global Private Limited,"308, Apra North Ex Plaza, Plot No.H-2, Netaji ...",India
4,S1-477299338,Pioneer Healthcare Private Limited,"Maharashtra, Mumbai, Sahakar Bhavan Kathabazar...",India


,entity_id,business_name,business_address,country
0,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US
1,S2-652827281,E U Pioberce Furniture LLC,"106 KENNEDY ROAD, LOUDOUN COUNTY, VA",US
2,S2-569534268,Central Hospitality,"666 VI-LILLY CIR, GROVE CITY, OH",US
3,S2-328900052,"Pioneer-Financial, LLC","4249 HCIKORY RD, CHESTERFIELD COUNTY, VA",US
4,S2-389998118,Jmn Advertising Public Limited,"RZ -102, B BLOCK, 2ND FOOR, NARSINGH GARDEN, K...",India


## 6. Build Pair-Level Dataset

Join each development pair with its corresponding S1 and S2/S3 records.

The resulting table will contain the raw fields needed to calculate
pair-level similarity features.

In [20]:
# Prepare S1 records
s1_features = s1_pair_records.rename(
    columns={
        "entity_id": "source1_entity_id",
        "business_name": "s1_business_name",
        "business_address": "s1_business_address",
        "country": "s1_country"
    }
)

# Prepare candidate records
candidate_features = candidate_pair_records.rename(
    columns={
        "entity_id": "candidate_entity_id",
        "business_name": "candidate_business_name",
        "business_address": "candidate_business_address",
        "country": "candidate_country"
    }
)

# Merge S1 information
pair_dataset = dev_pairs.merge(
    s1_features,
    on="source1_entity_id",
    how="left",
    validate="many_to_one"
)

# Merge candidate information
pair_dataset = pair_dataset.merge(
    candidate_features,
    on="candidate_entity_id",
    how="left",
    validate="many_to_one"
)

print("Pair dataset shape:", pair_dataset.shape)

display(pair_dataset.head())

Pair dataset shape: (35206, 9)


,source1_entity_id,candidate_entity_id,label,s1_business_name,s1_business_address,s1_country,candidate_business_name,candidate_business_address,candidate_country
0,S1-589179138,S3-570768070,1,Womens Health Gulf Specialists,"2720 Alder Ridge Drive, North Bend, OR",US,Womens Health Gulf (Specialists),"2720 Alder Ridge Drive, North Bend, Oregon",US
1,S1-721348248,S2-778835740,1,Good Infra,"F-269(B), Road No. 13 Vkia, Jaipur, Rajasthan",India,गुड इंफ्रा,"ROAD NO. 13 VKIA, JAIPUR, Rajasthan, F-269(B)",India
2,S1-841452359,S3-492698045,1,"Stringer, Black and Boyd","14514 Adkins Road, Carlinville, IL",US,"Stringer, Black and Boyd","14514 Adkrns Road, Cralinville, Illinois",US
3,S1-371936370,S3-638288652,0,Ridge Charter School,"1576 Support Avenue, Fort Cavazos, TX",US,Shimizu Réouck Twin LLC,"1929 162nd Ave, Ham Lake, Minnesota",US
4,S1-466333379,S3-506627624,1,Hitech Properties LLP,"Kp 3/710 K, Orchid Ppd Blooms Vattappara, Thir...",India,ഹൈടെക് പ്രോപ്പർട്ടീസ് എൽഎൽപി,"Kp 3/710 K, Orchid Ppd Blooms Vattappara, Thir...",India


In [21]:
assert len(pair_dataset) == len(dev_pairs)

required_columns = [
    "source1_entity_id",
    "candidate_entity_id",
    "s1_business_name",
    "s1_business_address",
    "s1_country",
    "candidate_business_name",
    "candidate_business_address",
    "candidate_country",
    "label"
]

missing_columns = [
    col for col in required_columns
    if col not in pair_dataset.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

print("✓ Pair dataset successfully constructed.")

✓ Pair dataset successfully constructed.


In [22]:
print("POSITIVE PAIR")
display(
    pair_dataset[pair_dataset["label"] == 1].head(5)
)

print("\nNEGATIVE PAIR")
display(
    pair_dataset[pair_dataset["label"] == 0].head(5)
)

POSITIVE PAIR


,source1_entity_id,candidate_entity_id,label,s1_business_name,s1_business_address,s1_country,candidate_business_name,candidate_business_address,candidate_country
0,S1-589179138,S3-570768070,1,Womens Health Gulf Specialists,"2720 Alder Ridge Drive, North Bend, OR",US,Womens Health Gulf (Specialists),"2720 Alder Ridge Drive, North Bend, Oregon",US
1,S1-721348248,S2-778835740,1,Good Infra,"F-269(B), Road No. 13 Vkia, Jaipur, Rajasthan",India,गुड इंफ्रा,"ROAD NO. 13 VKIA, JAIPUR, Rajasthan, F-269(B)",India
2,S1-841452359,S3-492698045,1,"Stringer, Black and Boyd","14514 Adkins Road, Carlinville, IL",US,"Stringer, Black and Boyd","14514 Adkrns Road, Cralinville, Illinois",US
4,S1-466333379,S3-506627624,1,Hitech Properties LLP,"Kp 3/710 K, Orchid Ppd Blooms Vattappara, Thir...",India,ഹൈടെക് പ്രോപ്പർട്ടീസ് എൽഎൽപി,"Kp 3/710 K, Orchid Ppd Blooms Vattappara, Thir...",India
5,S1-74837897,S3-959247066,1,Summit Initiative PC,"1531 Juniper Street, Bend, OR",US,Summit Initiative,"1531- Juniper St, Bend, Oregon",US



NEGATIVE PAIR


,source1_entity_id,candidate_entity_id,label,s1_business_name,s1_business_address,s1_country,candidate_business_name,candidate_business_address,candidate_country
3,S1-371936370,S3-638288652,0,Ridge Charter School,"1576 Support Avenue, Fort Cavazos, TX",US,Shimizu Réouck Twin LLC,"1929 162nd Ave, Ham Lake, Minnesota",US
6,S1-25983407,S3-227654588,0,Siva Science Center,"Room No.26D, Ground Floor Infopark Tbc, Jni St...",India,Sri Jay Marketing Pvt Ltd.,"G-905, Sector-17 B, Gurgaon, HR",India
8,S1-758288766,S3-429961399,0,Promoters Video Corp,"Hyderabad, Rajendranagar, Flat No 007, A Block...",India,Southern Managematt,"BR, C/o-at-flat No-404., Patna",India
9,S1-712770952,S3-291728652,0,Ess Traders Pvt Ltd,"G-15, Princearcade, 22-A, .Cathedral Road Chen...",India,अल्फा प्रोजेक्ट्स जनरल प्राइवेट लिमिटेड,"H.no 64 Estragen House, Jaipur, RJ",India
10,S1-204250181,S2-446338012,0,Great Software Private Limited,"45/5 Nandivli Panchanand Dombivli, Thane, Maha...",India,रियल प्रोड्यूसर एलएलपी,"DOOR NO 523 INGALI TAL HAT HATKANANGALE, HATKA...",India


## 7. Pair-Level Feature Engineering

Create interpretable similarity features for each S1–S2/S3 pair.

### Name features
- Normalized exact equality
- Character similarity
- Token Jaccard similarity

### Address features
- Normalized exact equality
- Character similarity
- Token Jaccard similarity

### Additional features
- Country equality
- Missing-name indicators
- Missing-address indicators

The features are computed independently for each candidate pair.

In [23]:
import rapidfuzz

print("RapidFuzz version:", rapidfuzz.__version__)

RapidFuzz version: 3.14.6


In [24]:
import re
import unicodedata

from rapidfuzz.fuzz import ratio


def normalize_text(value):
    """
    Lightweight normalization for Matcher V1.
    
    Keeps the raw data untouched and creates a comparison representation.
    """
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    # Unicode normalization
    value = unicodedata.normalize("NFKC", value)

    # Replace punctuation with spaces
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)

    # Collapse whitespace
    value = re.sub(r"\s+", " ", value).strip()

    return value


def token_jaccard(a, b):
    """
    Jaccard similarity between token sets.
    """
    a_tokens = set(normalize_text(a).split())
    b_tokens = set(normalize_text(b).split())

    if not a_tokens and not b_tokens:
        return 1.0

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def character_similarity(a, b):
    """
    Character-level similarity in [0, 1].
    """
    a_norm = normalize_text(a)
    b_norm = normalize_text(b)

    if not a_norm and not b_norm:
        return 1.0

    if not a_norm or not b_norm:
        return 0.0

    return ratio(a_norm, b_norm) / 100.0

In [25]:
def build_pair_features(df):
    features = pd.DataFrame(index=df.index)

    # -------------------------
    # Name normalization
    # -------------------------
    s1_name = df["s1_business_name"].map(normalize_text)
    cand_name = df["candidate_business_name"].map(normalize_text)

    features["name_exact"] = (
        (s1_name == cand_name) &
        (s1_name != "")
    ).astype(int)

    features["name_char_similarity"] = [
        character_similarity(a, b)
        for a, b in zip(
            df["s1_business_name"],
            df["candidate_business_name"]
        )
    ]

    features["name_token_jaccard"] = [
        token_jaccard(a, b)
        for a, b in zip(
            df["s1_business_name"],
            df["candidate_business_name"]
        )
    ]

    # -------------------------
    # Address
    # -------------------------
    s1_address = df["s1_business_address"].map(normalize_text)
    cand_address = df["candidate_business_address"].map(normalize_text)

    features["address_exact"] = (
        (s1_address == cand_address) &
        (s1_address != "")
    ).astype(int)

    features["address_char_similarity"] = [
        character_similarity(a, b)
        for a, b in zip(
            df["s1_business_address"],
            df["candidate_business_address"]
        )
    ]

    features["address_token_jaccard"] = [
        token_jaccard(a, b)
        for a, b in zip(
            df["s1_business_address"],
            df["candidate_business_address"]
        )
    ]

    # -------------------------
    # Country
    # -------------------------
    features["country_match"] = (
        df["s1_country"].fillna("").str.lower()
        ==
        df["candidate_country"].fillna("").str.lower()
    ).astype(int)

    # -------------------------
    # Missingness
    # -------------------------
    features["s1_name_missing"] = (
        df["s1_business_name"].isna() |
        (df["s1_business_name"].astype(str).str.strip() == "")
    ).astype(int)

    features["candidate_name_missing"] = (
        df["candidate_business_name"].isna() |
        (df["candidate_business_name"].astype(str).str.strip() == "")
    ).astype(int)

    features["s1_address_missing"] = (
        df["s1_business_address"].isna() |
        (df["s1_business_address"].astype(str).str.strip() == "")
    ).astype(int)

    features["candidate_address_missing"] = (
        df["candidate_business_address"].isna() |
        (df["candidate_business_address"].astype(str).str.strip() == "")
    ).astype(int)

    return features

In [26]:
X_dev = build_pair_features(pair_dataset)
y_dev = pair_dataset["label"].astype(int)

print("Feature matrix shape:", X_dev.shape)
print("\nFeatures:")
print(list(X_dev.columns))

display(X_dev.head())

Feature matrix shape: (35206, 11)

Features:
['name_exact', 'name_char_similarity', 'name_token_jaccard', 'address_exact', 'address_char_similarity', 'address_token_jaccard', 'country_match', 's1_name_missing', 'candidate_name_missing', 's1_address_missing', 'candidate_address_missing']


,name_exact,name_char_similarity,name_token_jaccard,address_exact,address_char_similarity,address_token_jaccard,country_match,s1_name_missing,candidate_name_missing,s1_address_missing,candidate_address_missing
0,1,1.000000,1.0,0,0.947368,0.75,1,0,0,0,0
1,0,0.105263,0.0,0,0.800000,1.00,1,0,0,0,0
2,1,1.000000,1.0,0,0.857143,0.25,1,0,0,0,0
3,0,0.279070,0.0,0,0.323529,0.00,1,0,0,0,0
4,0,0.086957,0.0,0,0.890511,0.75,1,0,0,0,0


## 8. Entity-Level Matcher Development Split

Split the development S1 entities into training and validation groups.

The split is performed at the S1 entity level so that pairs belonging to the
same S1 entity cannot appear in both training and validation.

This prevents pair-level leakage during Matcher V1 evaluation.

In [27]:
from sklearn.model_selection import train_test_split

pair_s1_ids = pair_dataset["source1_entity_id"].unique()

matcher_train_s1, matcher_valid_s1 = train_test_split(
    pair_s1_ids,
    test_size=0.20,
    random_state=RANDOM_STATE
)

matcher_train_s1 = set(matcher_train_s1)
matcher_valid_s1 = set(matcher_valid_s1)

print("Matcher training S1:", len(matcher_train_s1))
print("Matcher validation S1:", len(matcher_valid_s1))

assert not (matcher_train_s1 & matcher_valid_s1)

print("✓ No S1 entity appears in both matcher partitions.")

Matcher training S1: 3848
Matcher validation S1: 963
✓ No S1 entity appears in both matcher partitions.


In [28]:
train_mask = pair_dataset["source1_entity_id"].isin(
    matcher_train_s1
)

valid_mask = pair_dataset["source1_entity_id"].isin(
    matcher_valid_s1
)

X_train = X_dev.loc[train_mask].copy()
y_train = y_dev.loc[train_mask].copy()

X_valid = X_dev.loc[valid_mask].copy()
y_valid = y_dev.loc[valid_mask].copy()

print("Training pairs:", len(X_train))
print("Validation pairs:", len(X_valid))

print("\nTraining labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_valid.value_counts())

Training pairs: 28250
Validation pairs: 6956

Training labels:
label
0    14130
1    14120
Name: count, dtype: int64

Validation labels:
label
1    3483
0    3473
Name: count, dtype: int64


## 9. Matcher V1 — Logistic Regression Baseline

Train a simple binary classifier on the pair-level similarity features.

The model predicts:

- 1 → S1 and candidate are likely the same business
- 0 → S1 and candidate are likely different businesses

The model outputs probabilities so that the decision threshold can be tuned
for the precision-heavy F0.5 evaluation metric.

In [29]:
from sklearn.linear_model import LogisticRegression

matcher_v1 = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

matcher_v1.fit(
    X_train,
    y_train
)

print("✓ Matcher V1 trained.")

✓ Matcher V1 trained.


In [30]:
valid_probabilities = matcher_v1.predict_proba(
    X_valid
)[:, 1]

print("Validation probabilities:")
print(valid_probabilities[:10])

Validation probabilities:
[0.03381944 1.         0.00335804 0.99939265 0.98889331 0.99999974
 0.00249534 0.02820768 0.00261699 0.9999907 ]


In [31]:
thresholds = np.arange(
    0.10,
    0.96,
    0.05
)

threshold_results = []

for threshold in thresholds:

    predictions = (
        valid_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        predictions,
        zero_division=0
    )

    f05 = fbeta_score(
        y_valid,
        predictions,
        beta=0.5,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f0.5": f05
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(threshold_results)

,threshold,precision,recall,f0.5
0,0.10,0.979713,0.998277,0.983370
1,0.15,0.986659,0.997990,0.988905
2,0.20,0.991158,0.997703,0.992460
3,0.25,0.994559,0.997129,0.995072
4,0.30,0.995410,0.996268,0.995582
5,0.35,0.995402,0.994545,0.995231
6,0.40,0.996260,0.994258,0.995859
7,0.45,0.997118,0.993396,0.996372
8,0.50,0.997978,0.991961,0.996769
9,0.55,0.998554,0.991100,0.997054


## 10. Entity-Level Validation

Evaluate Matcher V1 at the S1-entity level.

For each S1 entity:

1. Convert candidate probabilities into predicted matched IDs.
2. Compare the predicted set against the ground-truth matched IDs.
3. Calculate F0.5 for each S1 entity.
4. Macro-average across validation S1 entities.

This better reflects the actual challenge evaluation structure.

In [32]:
def entity_level_macro_f05(
    pair_df,
    probabilities,
    threshold,
    gt_lookup
):
    temp = pair_df.copy()

    temp["probability"] = probabilities
    temp["predicted"] = (
        temp["probability"] >= threshold
    )

    entity_scores = []

    for s1_id, group in temp.groupby("source1_entity_id"):

        predicted_ids = set(
            group.loc[
                group["predicted"],
                "candidate_entity_id"
            ]
        )

        true_ids = gt_lookup.get(
            s1_id,
            set()
        )

        tp = len(predicted_ids & true_ids)
        fp = len(predicted_ids - true_ids)
        fn = len(true_ids - predicted_ids)

        if tp + fp == 0:
            precision = 0.0
        else:
            precision = tp / (tp + fp)

        if tp + fn == 0:
            recall = 1.0
        else:
            recall = tp / (tp + fn)

        if precision == 0 and recall == 0:
            f05 = 0.0
        else:
            f05 = (
                1.25 * precision * recall
                / (0.25 * precision + recall)
            )

        entity_scores.append({
            "source1_entity_id": s1_id,
            "precision": precision,
            "recall": recall,
            "f0.5": f05,
            "num_predicted": len(predicted_ids),
            "num_true": len(true_ids)
        })

    scores = pd.DataFrame(entity_scores)

    return scores["f0.5"].mean(), scores

In [33]:
valid_gt_lookup = {}

for _, row in dev_gt[
    dev_gt["source1_entity_id"].isin(matcher_valid_s1)
].iterrows():

    matched = row["matched_entity_ids"].strip()

    if matched == "":
        valid_gt_lookup[row["source1_entity_id"]] = set()
    else:
        valid_gt_lookup[row["source1_entity_id"]] = set(
            matched.split(",")
        )

print("Validation S1 entities in GT:", len(valid_gt_lookup))

Validation S1 entities in GT: 963


In [34]:
entity_f05_060, entity_scores_060 = entity_level_macro_f05(
    pair_dataset.loc[valid_mask],
    valid_probabilities,
    threshold=0.60,
    gt_lookup=valid_gt_lookup
)

print(
    f"Entity-level macro F0.5 @ 0.60: "
    f"{entity_f05_060:.6f}"
)

display(entity_scores_060.head())

Entity-level macro F0.5 @ 0.60: 0.980818


,source1_entity_id,precision,recall,f0.5,num_predicted,num_true
0,S1-100162607,1.0,1.0,1.0,2,2
1,S1-100792781,1.0,1.0,1.0,6,6
2,S1-102654924,1.0,1.0,1.0,3,3
3,S1-105063663,1.0,1.0,1.0,3,3
4,S1-106771351,1.0,1.0,1.0,3,3


## 11. Entity-Level Threshold Tuning

Evaluate Matcher V1 across multiple probability thresholds using
entity-level macro F0.5.

The threshold will be selected using the validation entities only.

In [35]:
entity_threshold_results = []

for threshold in np.arange(0.30, 0.91, 0.05):

    score, _ = entity_level_macro_f05(
        pair_dataset.loc[valid_mask],
        valid_probabilities,
        threshold=threshold,
        gt_lookup=valid_gt_lookup
    )

    entity_threshold_results.append({
        "threshold": threshold,
        "entity_macro_f0.5": score
    })

entity_threshold_results = pd.DataFrame(
    entity_threshold_results
)

display(entity_threshold_results)

,threshold,entity_macro_f0.5
0,0.30,0.980491
1,0.35,0.979826
2,0.40,0.980450
3,0.45,0.980977
4,0.50,0.981052
5,0.55,0.981152
6,0.60,0.980818
7,0.65,0.980586
8,0.70,0.979659
9,0.75,0.978359


In [36]:
best_entity_threshold = (
    entity_threshold_results
    .loc[
        entity_threshold_results["entity_macro_f0.5"].idxmax()
    ]
)

print("Best threshold:")
print(best_entity_threshold)

Best threshold:
threshold            0.550000
entity_macro_f0.5    0.981152
Name: 5, dtype: float64


## Candidate Pair Integration — Teammate Blocking Output

This section inspects the candidate pairs generated by the blocking/retrieval pipeline before integrating them with the ML matcher.

The candidate file is large, so it will be inspected in chunks rather than loaded completely into memory.

In [37]:
from pathlib import Path

CANDIDATE_PATH = (
    Path("..")
    / "teammate_data"
    / "candidate_pairs.tsv"
    / "candidate_pairs.tsv"
)

print("Candidate file:")
print(CANDIDATE_PATH.resolve())

print("\nExists:", CANDIDATE_PATH.exists())

if CANDIDATE_PATH.exists():
    print(
        "Size (GB):",
        round(CANDIDATE_PATH.stat().st_size / (1024**3), 2)
    )

Candidate file:
C:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\teammate_data\candidate_pairs.tsv\candidate_pairs.tsv

Exists: True
Size (GB): 2.99


In [38]:
candidate_sample = pd.read_csv(
    CANDIDATE_PATH,
    sep="\t",
    nrows=10,
    dtype=str
)

print("Columns:")
print(candidate_sample.columns.tolist())

print("\nSample shape:", candidate_sample.shape)

display(candidate_sample)

Columns:
['source1_entity_id', 'candidate_entity_ids']

Sample shape: (10, 2)


,source1_entity_id,candidate_entity_ids
0,S1-925783039,"S2-157377754,S2-517291332,S3-997698194"
1,S1-773889195,"S2-101629612,S2-115549519,S2-133154141,S2-2136..."
2,S1-377745466,"S2-144682432,S2-20283464,S2-203037501,S2-25343..."
3,S1-133037285,"S2-103437512,S2-103613846,S2-116829652,S2-1227..."
4,S1-755362802,"S2-589948701,S2-702005781,S3-139623733,S3-1717..."
5,S1-851869949,"S2-729231307,S2-801813753,S3-807524060"
6,S1-785847572,"S2-136041364,S2-164190611,S2-177968913,S2-1804..."
7,S1-27541239,"S2-110242163,S2-315097146,S2-427262176,S2-5635..."
8,S1-629417405,"S2-522538408,S2-928426462,S3-613837325,S3-7861..."
9,S1-22305073,"S2-709758118,S3-183589405,S3-221070872,S3-2259..."


In [39]:
# Inspect the full candidate file without loading it into memory

total_rows = 0
total_candidate_ids = 0
empty_candidate_rows = 0
unique_s1_ids = set()

for chunk in pd.read_csv(
    CANDIDATE_PATH,
    sep="\t",
    dtype=str,
    chunksize=100_000
):
    total_rows += len(chunk)

    unique_s1_ids.update(chunk["source1_entity_id"])

    candidate_lists = chunk["candidate_entity_ids"].fillna("")

    empty_candidate_rows += (candidate_lists.str.strip() == "").sum()

    total_candidate_ids += (
        candidate_lists
        .str.split(",")
        .str.len()
        .sum()
    )

print("Candidate file rows:", f"{total_rows:,}")
print("Unique S1 entities:", f"{len(unique_s1_ids):,}")
print("Total candidate IDs:", f"{total_candidate_ids:,}")
print("S1 rows with no candidates:", f"{empty_candidate_rows:,}")

if total_rows > 0:
    print(
        "Average candidates per S1:",
        round(total_candidate_ids / total_rows, 2)
    )

Candidate file rows: 2,206,821
Unique S1 entities: 2,206,821
Total candidate IDs: 246,556,279
S1 rows with no candidates: 22,783
Average candidates per S1: 111.72


In [40]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
FLAT_PATH = PROJECT_ROOT / "teammate_data" / "candidate_pairs_flat.tsv"

print("Flat candidate path:")
print(FLAT_PATH)

print("\nExists:", FLAT_PATH.exists())

if FLAT_PATH.exists():
    print("Size (MB):", round(FLAT_PATH.stat().st_size / (1024**2), 2))

    sample = pd.read_csv(
        FLAT_PATH,
        sep="\t",
        nrows=5,
        dtype=str
    )

    print("\nColumns:")
    print(sample.columns.tolist())

    print("\nSample:")
    display(sample)

Flat candidate path:
c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\teammate_data\candidate_pairs_flat.tsv

Exists: True
Size (MB): 3.68

Columns:
['source1_entity_id', 'candidate_entity_id']

Sample:


,source1_entity_id,candidate_entity_id
0,S1-925783039,S2-157377754
1,S1-925783039,S2-517291332
2,S1-925783039,S3-997698194
3,S1-773889195,S2-101629612
4,S1-773889195,S2-115549519


In [41]:
# Check the actual size and number of candidate pairs

candidate_df = pd.read_csv(
    FLAT_PATH,
    sep="\t",
    dtype=str
)

print("Candidate pairs loaded:", len(candidate_df))
print("Unique S1 entities:", candidate_df["source1_entity_id"].nunique())
print("Unique candidate entities:", candidate_df["candidate_entity_id"].nunique())

print("\nFirst 5 rows:")
display(candidate_df.head())

print("\nLast 5 rows:")
display(candidate_df.tail())

Candidate pairs loaded: 149537
Unique S1 entities: 916
Unique candidate entities: 140971

First 5 rows:


,source1_entity_id,candidate_entity_id
0,S1-925783039,S2-157377754
1,S1-925783039,S2-517291332
2,S1-925783039,S3-997698194
3,S1-773889195,S2-101629612
4,S1-773889195,S2-115549519



Last 5 rows:


,source1_entity_id,candidate_entity_id
149532,S1-119783341,S3-991953131
149533,S1-119783341,S3-993520856
149534,S1-119783341,S3-99618323
149535,S1-119783341,S3-997442655
149536,S1-119783341,S3-998642976


In [42]:
from pathlib import Path

FULL_CANDIDATE_PATH = (
    PROJECT_ROOT
    / "teammate_data"
    / "candidate_pairs_flat_FULL.tsv"
)

print("Full candidate file:")
print(FULL_CANDIDATE_PATH)

print("\nExists:", FULL_CANDIDATE_PATH.exists())

if FULL_CANDIDATE_PATH.exists():
    size_gb = FULL_CANDIDATE_PATH.stat().st_size / (1024**3)
    print("Size (GB):", round(size_gb, 2))

Full candidate file:
c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\teammate_data\candidate_pairs_flat_FULL.tsv

Exists: True
Size (GB): 5.92


In [43]:
# Verify the full candidate file using chunked reading

CHUNK_SIZE = 500_000

total_rows = 0
unique_s1 = set()

for chunk in pd.read_csv(
    FULL_CANDIDATE_PATH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE
):
    total_rows += len(chunk)
    unique_s1.update(chunk["source1_entity_id"].unique())

print("Total candidate pairs:", f"{total_rows:,}")
print("Unique S1 entities:", f"{len(unique_s1):,}")

Total candidate pairs: 246,533,496
Unique S1 entities: 2,184,038


In [44]:
# Candidate count distribution per S1
# Process the 5.92 GB file in chunks.

candidate_counts = {}

for chunk in pd.read_csv(
    FULL_CANDIDATE_PATH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE
):
    counts = chunk["source1_entity_id"].value_counts()

    for s1_id, count in counts.items():
        candidate_counts[s1_id] = (
            candidate_counts.get(s1_id, 0) + count
        )

candidate_count_series = pd.Series(candidate_counts)

print("S1 entities with candidates:", f"{len(candidate_count_series):,}")
print("Total candidate pairs:", f"{candidate_count_series.sum():,}")

print("\nCandidate count statistics:")
display(candidate_count_series.describe())

print("\nS1 entities with zero candidates cannot appear in this file.")
print("Maximum candidates for one S1:", candidate_count_series.max())

S1 entities with candidates: 2,184,038
Total candidate pairs: 246,533,496

Candidate count statistics:


count    2.184038e+06
mean     1.128797e+02
std      2.116487e+02
min      1.000000e+00
25%      8.000000e+00
50%      3.700000e+01
75%      1.150000e+02
max      4.536000e+03
dtype: float64


S1 entities with zero candidates cannot appear in this file.
Maximum candidates for one S1: 4536


In [45]:
print("Training S1:", len(train_s1_ids))
print("Validation S1:", len(valid_s1_ids))

Training S1: 1765456
Validation S1: 441365


In [46]:
# Check whether the development split was saved somewhere

from pathlib import Path

for path in Path(PROJECT_ROOT).rglob("*"):
    if path.is_file() and any(
        key in path.name.lower()
        for key in ["split", "train_s1", "valid_s1", "development"]
    ):
        print(path)

c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\.venv\Lib\site-packages\IPython\core\splitinput.py
c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\.venv\Lib\site-packages\IPython\core\__pycache__\splitinput.cpython-312.pyc
c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\.venv\Lib\site-packages\jedi\third_party\typeshed\stubs\netaddr\netaddr\contrib\subnet_splitter.pyi
c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\.venv\Lib\site-packages\jedi\third_party\typeshed\stubs\reportlab\reportlab\lib\textsplit.pyi
c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\.venv\Lib\site-packages\numpy\f2py\tests\test_semicolon_split.py
c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\.venv\Lib\site-packages\numpy\f2py\tests\__pycache__\test_semicolon_split.cpython-312.pyc
c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource\.venv\Lib\site-packages\pandas\tests\strings\test_split_partition.py
c:\Users\sneha\Amazon-ML-Challenge-2026\student_resource

In [47]:
print("Training S1:", len(train_s1_ids))
print("Validation S1:", len(valid_s1_ids))

Training S1: 1765456
Validation S1: 441365
